# Homework 1

**Name:** Revanth Challa

This notebook solves the three problems from Homework 1. Each of the three data files (`Hw1_P1_Text.txt`, `Hw1_P2.json`, and `Hw1_P3.json`) is expected to sit in the same directory as this notebook when you run it (in Colab, upload them with the file browser first).

## Problem 1: Python I/O Warmup

Read `Hw1_P1_Text.txt` one line at a time, reverse each line (the lyrics are stored backwards), and print the results.

In [ ]:
# Problem 1
with open('Hw1_P1_Text.txt', 'r') as f:
    for line in f:
        # strip the trailing newline, reverse the remaining characters,
        # and print. Blank lines reverse to blank lines, so stanza
        # breaks are preserved automatically.
        reversed_line = line.rstrip('\n')[::-1]
        print(reversed_line)

## Problem 2: Election Polling

Read all 10,000 votes from `Hw1_P2.json`, compute the true level of support for Alice, then run five independent 100-voter polls (sampling without replacement) and report each estimate.

In [ ]:
# Problem 2
import json
import random

with open('Hw1_P2.json', 'r') as f:
    data = json.load(f)

# The stored strings occasionally have a leading space (e.g. ' Alice'),
# so normalize before counting.
votes = [v.strip() for v in data['Votes']]
total_voters = len(votes)

# Ground-truth support for Alice, as a percentage.
true_alice_count = sum(1 for v in votes if v == 'Alice')
true_alice_pct = 100.0 * true_alice_count / total_voters
print(f'Ground-truth support for Alice: {true_alice_pct:.2f}% ({true_alice_count}/{total_voters})')

# Five independent polls of 100 voters each, sampled without replacement.
num_polls = 5
sample_size = 100
print()
for i in range(1, num_polls + 1):
    shuffled = votes.copy()
    random.shuffle(shuffled)
    poll_sample = shuffled[:sample_size]
    poll_alice_pct = 100.0 * sum(1 for v in poll_sample if v == 'Alice') / sample_size
    print(f'Poll {i}: support for Alice = {poll_alice_pct:.0f}%')

### Comments on polling strategy

The biggest lever for improving accuracy is **increasing the sample size**. A poll of 100 voters has a margin of error of roughly $\pm 10$ percentage points (the standard error for a proportion near 0.5 with $n=100$ is $\sqrt{0.5\cdot 0.5/100} = 0.05$, i.e. 5 pp, so a 95% interval is about $\pm 10$ pp). Going to $n=1000$ cuts that roughly in half.

A few additional strategies:

- **Average across many independent polls** (or equivalently, treat multiple pollsters as one aggregated sample) to shrink the variance of the estimate.
- **Stratified sampling**: split the population into groups likely to vote differently (age, region, party registration) and sample proportionally from each. This reduces variance relative to simple random sampling when strata are informative.
- **Careful randomization** so the sample really is representative — avoid convenience or self-selected samples, which introduce bias no amount of larger $n$ can fix.

## Problem 3: Data Visualization and Analysis

Load the 20 $(X, Y)$ points from `Hw1_P3.json`, plot them, estimate the integral over $[0, 2]$ using (a) the trapezoidal rule directly on the samples, and (b) a cubic spline interpolant fed into `scipy.integrate.quad`.

In [ ]:
# Problem 3
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate, interpolate

with open('Hw1_P3.json', 'r') as f:
    p3 = json.load(f)

x = np.array(p3['X Values'])
y = np.array(p3['Y Values'])

# 1) Scatter plot of the raw data.
plt.figure(figsize=(8, 5))
plt.scatter(x, y, color='tab:blue', label='Measured data')
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Problem 3: Experimental (X, Y) data')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# 2) Trapezoidal rule directly on the samples.
trapezoid_area = integrate.trapezoid(y, x)

# 3) Cubic spline interpolation, then integrate with quad.
spline = interpolate.make_interp_spline(x, y, k=3)
spline_area, spline_err = integrate.quad(spline, x[0], x[-1])

print(f'Estimated area (trapezoidal rule):    {trapezoid_area:.6f}')
print(f'Estimated area (cubic spline + quad): {spline_area:.6f}')
print(f'  (quad reported absolute error estimate: {spline_err:.2e})')

# Overlay the spline on the data for a visual sanity check.
x_dense = np.linspace(x[0], x[-1], 400)
plt.figure(figsize=(8, 5))
plt.scatter(x, y, color='tab:blue', label='Measured data')
plt.plot(x_dense, spline(x_dense), color='tab:orange', label='Cubic spline interpolant')
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Problem 3: Data with cubic spline interpolant')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

**Discussion.** Both methods integrate a function that passes exactly through the sampled points; they differ in how they connect the samples. The trapezoidal rule uses straight-line segments between neighbors, while the cubic spline matches value, first, and second derivatives at each interior sample, producing a smoother curve. When the true underlying process is smooth, the spline estimate is usually the more accurate of the two; when the data is sparse or noisy, the two estimates can disagree noticeably, and their spread gives a rough sense of the uncertainty in the area estimate.